# BSCP sack-train-ml — Compile YOLOv11s → HEF (Colab)

> Loom Oracle (AI) · 2026-06-20 · คู่กับ `notebooks/train_run.ipynb` — อันนั้นเทรน, อันนี้ compile HEF
> pipeline: `.onnx → parse(.har) → optimize(int8) → compile(.hef) → emulator eval → meta.yaml`

## ▶ To start: Runtime → Change runtime type → **High-RAM** (GPU T4 ถ้ามี) → Run all ทีละ section

รันบน Colab (Ubuntu x86_64) แทน WSL. **อ่าน 2 กล่องเตือนด้านล่างก่อนเริ่ม**

---
### ⚠️ เตือน 1 — เวอร์ชันต้องตรงกับ edge
edge รัน **HailoRT 4.20.0**. HEF ที่ compile ออกมา **ต้องโหลดบน 4.20.0 ได้**.
- ตรงกับ 4.20.0 แบบชัวร์ = **DFC 3.30.0 + model zoo 2.14.0** (Hailo Suite 2025-01)
- เธอใช้ **model zoo 2.18.0** (คู่กับ DFC 3.33.1, ใหม่กว่า) → **ต้องทดสอบว่า HEF โหลดบน Pi จริง** (ขั้นสุดท้าย) ถ้าไม่ได้ค่อยถอยมา 2.14/3.30

### ⚠️ เตือน 2 — โมเดลเราเป็น custom 2 คลาส (Person, Sack) ไม่ใช่ COCO
`hailomz` มี config ของ yolov11s เป็น COCO 80 คลาส. เราต้องปรับ **จำนวนคลาส = 2**, end-node names, และ NMS config ให้ตรง — จุดที่มี `# TODO` คือจุดที่ต้องปรับตามโมเดลจริง


## 0) Config — แก้ค่าตรงนี้ที่เดียว


In [ ]:
# ===== แก้ค่าตรงนี้ =====
DFC_VERSION   = '3.33.1'      # ให้ตรงกับ model zoo ของเธอ (2.18.0 -> 3.33.x). 4.20.0-safe = 3.30.0
MZ_VERSION    = '2.18.0'      # hailo_model_zoo ที่เธอมี
HW_ARCH       = 'hailo8l'
MODEL_NAME    = 'yolov11s'
NUM_CLASSES   = 2             # Person, Sack
CLASS_NAMES   = ['Person', 'Sack']
INPUT_SHAPE   = [640, 640, 3]

# ไฟล์ที่ต้องอัปขึ้น Colab (ดูขั้น 1)
ONNX_PATH     = '/content/model.onnx'                 # yolov11s ของเรา
CALIB_DIR     = '/content/calib'                      # >=256 รูป representative
WORK          = '/content/work'
OUT_HEF       = f'{WORK}/{MODEL_NAME}.hef'

import os; os.makedirs(WORK, exist_ok=True)
print('config ok ->', MODEL_NAME, NUM_CLASSES, 'classes, target', HW_ARCH)

## 0.1) Sync repo (เหมือน train_run.ipynb)
clone `sack-train-ml` @ main — เผื่อใช้ helper `sack_train_ml.hailo_pipeline.compile_hef` หรือ `scripts/` ในอนาคต (ตอนนี้ scripts ยัง stub, โน้ตบุ๊กนี้รัน hailomz ตรง ๆ)


In [ ]:
import subprocess, os
REPO_DIR = '/content/sack-train-ml'
REPO_URL = 'https://github.com/pitikorn-pam/sack-train-ml.git'
if os.path.exists(REPO_DIR):
    subprocess.run(['git','-C',REPO_DIR,'fetch','origin'], check=True)
    subprocess.run(['git','-C',REPO_DIR,'reset','--hard','origin/main'], check=True)
else:
    subprocess.run(['git','clone','--branch','main',REPO_URL,REPO_DIR], check=True)
GIT_SHA = subprocess.check_output(['git','-C',REPO_DIR,'rev-parse','--short','HEAD'],text=True).strip()
print('repo @', GIT_SHA)

## 1) อัปไฟล์ขึ้น Colab
ต้องมี 3 อย่าง:
1. **DFC whl** (`hailo_dataflow_compiler-3.33.x-...whl`) — โหลดจาก Hailo Developer Zone (gated)
2. **model zoo whl** (`hailo_model_zoo-2.18.0-py3-none-any.whl`) — ที่เธอมีแล้ว
3. **model.onnx** + **โฟลเดอร์ calibration** (รูปจริงจาก edge ≥256 รูป)

แนะนำ: เก็บไว้ใน Google Drive แล้ว mount จะเร็วกว่าอัปทุกครั้ง


In [ ]:
# ทางเลือก A: mount Google Drive (แนะนำ — ไม่ต้องอัปซ้ำทุก session)
from google.colab import drive
drive.mount('/content/drive')
# แล้ว copy whl/onnx/calib มาที่ /content เช่น:
# !cp '/content/drive/MyDrive/hailo/'*.whl /content/
# !cp '/content/drive/MyDrive/hailo/model.onnx' /content/model.onnx
# !cp -r '/content/drive/MyDrive/hailo/calib' /content/calib

In [ ]:
# ทางเลือก B: อัปจากเครื่องตรง ๆ (ไฟล์เล็ก)
# from google.colab import files; files.upload()

import glob
print('DFC whl :', glob.glob('/content/hailo_dataflow_compiler*.whl'))
print('MZ  whl :', glob.glob('/content/hailo_model_zoo*.whl'))
print('onnx    :', os.path.exists(ONNX_PATH))
print('calib n :', len(glob.glob(f'{CALIB_DIR}/*')))

## 2) Runtime check (High-RAM)
compile กิน RAM 12–32GB และใช้ CPU เป็นหลัก (GPU แค่ช่วยขั้น optimize ให้เร็วขึ้น)


In [ ]:
import subprocess, shutil
print('--- RAM ---'); print(subprocess.run(['free','-h'],capture_output=True,text=True).stdout)
print('--- GPU (optional) ---')
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],capture_output=True,text=True).stdout or 'no GPU (CPU-only compile, ช้าแต่ได้)')
import sys; print('python', sys.version.split()[0], '(DFC ต้องการ 3.10/3.11/3.12)')

## 3) System deps + cuDNN downgrade
Colab default cuDNN 9 ทำ DFC พัง — ต้อง downgrade เป็น cuDNN 8.9.7 + pin numpy/scipy (ตามคู่มือ Hailo Community)
> ถ้ารัน CPU-only ล้วน ข้ามส่วน cuDNN ได้ แต่ pin numpy/scipy ไว้เหมือนเดิม


In [ ]:
!sudo apt-get -qq update
!sudo apt-get -qq install -y python3-dev python3-tk libfuse2 graphviz libgraphviz-dev >/dev/null
# pin ที่ DFC ต้องการ (กัน error 'failed to initialize intent(inout)')
!pip -q install numpy==1.23.3 scipy==1.10.1
# cuDNN/CUDA สำหรับ GPU path (ข้ามได้ถ้า CPU-only)
# !pip -q install tensorrt==8.6.1
print('deps ok')

## 4) สร้าง venv + ติดตั้ง DFC และ model zoo
DFC ต้องอยู่ใน virtualenv (กัน error แปลก ๆ). เราจะเรียกผ่าน `!hailo_venv/bin/...`


In [ ]:
!pip -q install virtualenv
!virtualenv -p python3 /content/hailo_venv
VENV='/content/hailo_venv/bin'
!{VENV}/pip install --upgrade pip wheel >/dev/null
# ติดตั้ง DFC ก่อน แล้วตามด้วย model zoo
!{VENV}/pip install /content/hailo_dataflow_compiler*.whl
!{VENV}/pip install /content/hailo_model_zoo*.whl

In [ ]:
# verify เวอร์ชัน — ต้องตรงกับที่ตั้งใจ และเข้ากับ HailoRT 4.20.0 ของ edge
!/content/hailo_venv/bin/hailo --version
!/content/hailo_venv/bin/hailomz --version

## 5) (ถ้ายังไม่มี onnx) export จาก .pt
เรามี onnx อยู่แล้ว (`1.0.0-83484090.onnx`) — ข้ามได้. ถ้าจะ export ใหม่ ต้อง static shape + opset 11


In [ ]:
# !pip -q install ultralytics
# !yolo export model=yolov11s.pt format=onnx imgsz=640 opset=11 dynamic=False simplify=True
print('ใช้ onnx ที่อัปไว้:', ONNX_PATH)

## 6) Parse: onnx → .har
`hailomz parse` แปลงกราฟเป็น Hailo IR. **จด end_node_names ที่มันพิมพ์** ไปใส่ meta.yaml
> `# TODO` custom: yolov11s.yaml ของ model zoo เป็น COCO — ถ้าคลาส/หัวโมเดลไม่ตรง parse จะเตือน ให้แก้ end-nodes/ยamlตาม


In [ ]:
VENV='/content/hailo_venv/bin'
!cd {WORK} && {VENV}/hailomz parse --hw-arch {HW_ARCH} --ckpt {ONNX_PATH} {MODEL_NAME}
# ผลได้: {WORK}/{MODEL_NAME}.har
# 👉 จด end_node_names ที่ parse พิมพ์ (yolov11s = 6 conv: 3 scale x {reg,cls}) ไปใส่:
#    (ก) nms_config['bbox_decoders'] ขั้น optimize   (ข) end_node_names ใน meta.yaml ขั้น 11

## 7) Calibration set
คุณภาพ calib = ตัวชี้เป็นชี้ตายของ recall (โดยเฉพาะคลาสน้อย = Person). ใช้ **รูปจริงจาก edge** ครอบคลุม กลางวัน/กลางคืน/ฝุ่น/บัง/มุมกล้องจริง ≥256 รูป
เราจะทำ manifest.csv + sha256 (ไปใส่ meta.yaml.dataset_hash)


In [ ]:
import glob, hashlib, csv
imgs = sorted(glob.glob(f'{CALIB_DIR}/*.jpg') + glob.glob(f'{CALIB_DIR}/*.png'))
assert len(imgs) >= 256, f'ต้องการ >=256 รูป ตอนนี้มี {len(imgs)} — เพิ่ม calib ก่อน'
man = f'{WORK}/calibration_manifest.csv'
h = hashlib.sha256()
with open(man,'w',newline='') as f:
    w = csv.writer(f)
    for p in imgs:
        b = open(p,'rb').read(); d = hashlib.sha256(b).hexdigest(); h.update(d.encode())
        w.writerow([p, d])
DATASET_HASH = h.hexdigest()
print('calib images:', len(imgs), '| dataset_hash:', DATASET_HASH[:16], '...')

## 8) Optimize (quantize → int8) — **3 อย่างที่ต้อง match edge contract**
จากการวิเคราะห์ contract ของ edge (`hailo_backend.py`) `alls` ต้องมี:
1. **`normalization([0,0,0],[255,255,255])`** — edge ป้อน uint8 ดิบ ไม่ /255 เอง → HEF ต้องหารในตัว ไม่งั้น activation ใหญ่ 255 เท่า = garbage เงียบ
2. **`nms_postprocess(... classes=2 ...)`** — edge คาด NMS บนชิป, output ต่อคลาส (N,5). yolov11s ของ model zoo = 80 คลาส → ต้อง override เป็น **2** ไม่งั้น decode ผิด = นับศูนย์
3. **score_threshold ≤ 0.25** — เพราะ runtime ต้องการ detection ช่วง [0.35,0.70) ให้รอดไปเส้น flagging; ตั้งสูงไป = ตัด flagging band ทิ้งเงียบ
> ⚠️ syntax `nms_postprocess` ขึ้นกับเวอร์ชัน DFC + ต้องใส่ชื่อ output layer (bbox decoders) จากขั้น parse — จุด `# TODO` ด้านล่าง


In [ ]:
import json
# NMS config สำหรับ yolov11 (ใช้ decode แบบ yolov8 ใน hailo MZ), 2 คลาส
nms_cfg = {
  'nms_scores_th': 0.20,          # ต่ำไว้ (<=0.25) กัน flagging band หาย
  'nms_iou_th': 0.7,
  'image_dims': [INPUT_SHAPE[0], INPUT_SHAPE[1]],
  'max_proposals_per_class': 50,  # = --max_det ของ edge
  'classes': NUM_CLASSES,          # 2 (Person, Sack) — สำคัญสุด
  'regression_length': 16,
  'background_removal': False,
  # 'bbox_decoders': [...]   # TODO เติมชื่อ layer จาก output ขั้น parse (yolov11s = 3 scale)
}
open(f'{WORK}/nms_config.json','w').write(json.dumps(nms_cfg, indent=2))

alls = f'''
normalization1 = normalization([0.0, 0.0, 0.0], [255.0, 255.0, 255.0])
quantization_param({MODEL_NAME}, precision_mode=int8_weights, per_channel=true)
post_quantization_optimization(finetune, policy=enabled, dataset_size=256)
nms_postprocess("{WORK}/nms_config.json", meta_arch=yolov8, engine=cpu)
'''
open(f'{WORK}/{MODEL_NAME}.alls','w').write(alls)
print(alls)
VENV='/content/hailo_venv/bin'
!cd {WORK} && {VENV}/hailomz optimize --hw-arch {HW_ARCH} \
   --har {WORK}/{MODEL_NAME}.har \
   --calib-path {CALIB_DIR} \
   --model-script {WORK}/{MODEL_NAME}.alls \
   {MODEL_NAME}
# ผลได้: {MODEL_NAME}_optimized.har  (# TODO ตรวจชื่อไฟล์ที่ออกมาจริง)

## 9) Compile → .hef


In [ ]:
VENV='/content/hailo_venv/bin'
!cd {WORK} && {VENV}/hailomz compile --hw-arch {HW_ARCH} \
   --har {WORK}/{MODEL_NAME}_optimized.har \
   {MODEL_NAME}
import glob; print('HEF:', glob.glob(f'{WORK}/*.hef'))

## 9.1) ✅ Verify HEF ตรง edge contract (จับ silent failure ก่อนเสียเวลา deploy)
เช็ค 3 อย่างจากผลวิเคราะห์: input ต้องเป็น **UINT8 640×640×3**, output ต้องเป็น **NMS**, จำนวนคลาส == **2**


In [ ]:
from hailo_platform import HEF
import glob
hef = HEF(glob.glob(f'{WORK}/*.hef')[0])
i = hef.get_input_vstream_infos()[0]
outs = hef.get_output_vstream_infos()
print('input  :', i.name, i.shape, i.format.type)     # คาด UINT8, (640,640,3)
print('outputs:', [(o.name, str(o.format.order)) for o in outs])
in_type = str(i.format.type)
assert 'UINT8' in in_type, f'❌ input ไม่ใช่ uint8 ({in_type}) → edge ป้อน uint8 จะ garbage. แก้ normalization ใน alls'
assert list(i.shape[:2]) == INPUT_SHAPE[:2], f'❌ input shape {i.shape} != {INPUT_SHAPE}'
is_nms = any('NMS' in str(o.format.order).upper() for o in outs)
assert is_nms, '❌ output ไม่ใช่ NMS → edge decode ไม่ได้. เช็ค nms_postprocess ใน alls'
print('✅ contract ผ่านเบื้องต้น (uint8 + shape + NMS). คลาส=2 ยืนยันอีกทีตอน eval/บน Pi)')

## 10) Eval ด้วย emulator (วัด int8 mAP โดยไม่ต้องมีชิป)
`--target emulator` รัน bit-exact บน CPU — เทียบ fp32 vs int8 ตาม gate ของ edge: mAP50 drop ≤3%, recall drop ≤5%


In [ ]:
VENV='/content/hailo_venv/bin'
# ต้องมี eval set (รูป+label). ปรับ --data-path ให้ตรง
!cd {WORK} && {VENV}/hailomz eval --hw-arch {HW_ARCH} --target emulator \
   --har {WORK}/{MODEL_NAME}_optimized.har {MODEL_NAME}   # --data-path /content/eval
# จด mAP50/precision/recall (int8) ไปใส่ meta.yaml + เทียบ fp32

## 11) สร้าง meta.yaml + ดาวน์โหลด HEF
meta.yaml ตาม contract ของ edge (`docs/model-pipeline.md`). runtime ฝั่ง Pi จะปฏิเสธถ้า sha256 ไม่ตรง


In [ ]:
import hashlib, glob, datetime
hef = glob.glob(f'{WORK}/*.hef')[0]
sha = hashlib.sha256(open(hef,'rb').read()).hexdigest()
meta = f'''model: {MODEL_NAME}
version: "colab-{{stamp}}"
source_commit: "83484090"   # git sha ของ sack-train-ml ตอน export onnx
dataset_hash: "{DATASET_HASH}"
hailort_version: "4.20.0"
dfc_version: "{DFC_VERSION}"
input_shape: {INPUT_SHAPE}
end_node_names: []            # TODO เติมจาก output ขั้น parse
quantization:
  mode: int8
  calibration_size: 256
  per_channel: true
accuracy:
  fp32: {{mAP50: null, precision: null, recall: null}}
  int8: {{mAP50: null, precision: null, recall: null}}   # TODO เติมจากขั้น eval
gates_passed: false           # TODO ตั้ง true เมื่อผ่าน gate
sha256: "{sha}"
'''
stamp = '2026-06-20'  # หรือใส่วันจริง
meta = meta.replace('{stamp}', stamp)
mp = hef + '.meta.yaml'
open(mp,'w').write(meta)
print(mp); print(meta)
from google.colab import files
files.download(hef); files.download(mp)

## 12) ✅ ทดสอบบน Pi (สำคัญสุด — พิสูจน์เวอร์ชันตรง)
scp HEF + meta ไป edge แล้วโหลด:
```bash
hailortcli fw-control identify          # ยืนยัน HAILO8L + HailoRT 4.20.0
hailortcli run yolov11s.hef             # ถ้าโหลด+รันได้ = เวอร์ชันเข้ากันได้
```
ถ้า error **"invalid compiled format"** = DFC/HailoRT ไม่ match → ถอยมาใช้ **model zoo 2.14.0 + DFC 3.30.0** แล้ว compile ใหม่

---
_Loom Oracle (AI) — refine ได้เรื่อย ๆ จุด `# TODO` คือที่ต้องปรับตามโมเดลจริง_
